In [ ]:
import torch
import sentencepiece as spm

eng_tokenizer = spm.SentencePieceProcessor()
eng_tokenizer.load("english_tokenizer.model")

hindi_tokenizer = spm.SentencePieceProcessor()
hindi_tokenizer.load("hindi_tokenizer.model")



PAD_ID = 8000
BOS_ID = 1
EOS_ID = 2


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
from transformer_scratch import build_transformer
import torch

max_seq_length = 61

model = build_transformer(
    8001,
    8001,
    max_seq_length,
    max_seq_length,
    100,
    3,
    10,
    0.1,
    1000
)

model.load_state_dict(torch.load("model.pt"))

model = model.to(device)
model.eval()


def translate(sentence, max_len=61):
    src_tokens = eng_tokenizer.encode(
        sentence,
        out_type=int,
        add_bos=True,
        add_eos=True
    )

    if len(src_tokens) < max_len:
        src_tokens += [PAD_ID] * (max_len - len(src_tokens))
    else:
        src_tokens = src_tokens[:max_len]
        src_tokens[-1] = EOS_ID

    src = torch.tensor(src_tokens).unsqueeze(0).to(device)

    src_mask = (src != PAD_ID).unsqueeze(1).unsqueeze(2)

    with torch.no_grad():
        encoder_output = model.encode(src, src_mask)

        decoder_input = torch.tensor([[BOS_ID]], device=device)

        for _ in range(max_len):
            seq_len = decoder_input.size(1)

            causal_mask = torch.tril(
                torch.ones(
                    (seq_len, seq_len),
                    dtype=torch.bool,
                    device=device
                )
            ).unsqueeze(0).unsqueeze(0)

            pad_mask = (decoder_input != PAD_ID).unsqueeze(1).unsqueeze(2)

            tgt_mask = causal_mask & pad_mask

            decoder_output = model.decode(
                encoder_output,
                src_mask,
                decoder_input,
                tgt_mask
            )

            logits = model.project(decoder_output)

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1
            )

            decoder_input = torch.cat(
                [decoder_input, next_token.unsqueeze(1)],
                dim=1
            )

            if next_token.item() == EOS_ID:
                break

    output_tokens = decoder_input.squeeze(0).tolist()

    filtered_tokens = [
        token for token in output_tokens
        if token not in [BOS_ID, EOS_ID, PAD_ID]
    ]

    return hindi_tokenizer.decode(filtered_tokens)


while True:
    sentence = input("\nEnter English Sentence: ")

    if sentence.lower() == "exit":
        break

    translation = translate(sentence)

    print(f"\nEnglish: {sentence}")
    print(f"Hindi  : {translation}")


English: This is an example
Hindi  : यह एक उदाहरण है

English: I am using Laptop
Hindi  : मैं लापटॉप कंप्यूटर इस्तेमाल कर रहा हूँ

English: This is an example
Hindi  : यह एक उदाहरण है
